In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns.fpgrowth import fpgrowth
from mlxtend.frequent_patterns import association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
df=pd.read_csv('datasets//mercc_cleaned.csv')

In [ ]:
# Set hyper-paramater for machine learning

MIN_SUPPORT = 0.001
MIN_THRESH = 0.01 # For report generation

In [ ]:
# Feature selection 

df = df[['session_id', 'event_id', 'c0_name', 'c0_id', 'c1_name', 'c1_id']]

# Bucketing based on user action

# Completed purchase - level 0
bask_comp_purchase_zero = df.loc[df['event_id'] == 'buy_comp', ['session_id','c0_name']]

# Completed purchase - level 1
bask_comp_purchase_one = df.loc[df['event_id'] == 'buy_comp', ['session_id','c1_name']]
bask_comp_purchase_one = bask_comp_purchase_one.loc[bask_comp_purchase_one['c1_name'] != 'unknown',:]

# Added to cart - level 0
bask_add_cart_zero = df.loc[df['event_id'] == 'item_add_to_cart_tap', ['session_id','c0_name']]

# Added to cart - level 1
bask_add_cart_one = df.loc[df['event_id'] == 'item_add_to_cart_tap', ['session_id','c1_name']]
bask_add_cart_one = bask_add_cart_one.loc[bask_add_cart_one['c1_name'] != 'unknown',:]

# Item view and Item like is also consideration, but it's low priority

In [5]:
def melt_dataset(basket_df, categ_name, groupby='session_id'):
    '''
        basket_df: Target dataframe
        categ_name: column name of the target category, must be in string
        group_by(Optional): column name of the grouping variable

        Take a dataset (must be in wide format) and column names as an input 
        then turn into long format ready for machine learning
    '''
    basket_df = basket_df.groupby(groupby)[categ_name].apply(list).to_list()

    trans_encoder = TransactionEncoder()

    trans_encoder.fit(basket_df)

    enctrans_basket_df = trans_encoder.transform(basket_df)

    return pd.DataFrame(enctrans_basket_df, columns=trans_encoder.columns_)


bask_comp_purchase_zero_proc = melt_dataset(bask_comp_purchase_zero, 'c0_name')
bask_comp_purchase_one_proc = melt_dataset(bask_comp_purchase_one, 'c1_name')

bask_add_cart_zero_proc = melt_dataset(bask_add_cart_zero, 'c0_name')
bask_add_cart_one_proc = melt_dataset(bask_add_cart_one, 'c1_name')

In [6]:
# Model Training
freq_bask_comp_purchase_zero = fpgrowth(bask_comp_purchase_zero_proc, min_support=MIN_SUPPORT, use_colnames=True)
freq_bask_comp_purchase_one = fpgrowth(bask_comp_purchase_one_proc, min_support=MIN_SUPPORT, use_colnames=True)

freq_bask_add_cart_zero = fpgrowth(bask_add_cart_zero_proc, min_support=MIN_SUPPORT, use_colnames=True)
freq_bask_add_cart_one = fpgrowth(bask_add_cart_one_proc, min_support=MIN_SUPPORT, use_colnames=True)

In [8]:
# Creating report

machine_dict = {
    "comp_purchase_zero": freq_bask_comp_purchase_zero,
    "comp_purchase_one": freq_bask_comp_purchase_one,
    "add_cart_zero": freq_bask_add_cart_zero,
    "add_cart_one": freq_bask_add_cart_one
}

for name, data in machine_dict.items():
    # Generate rules
    rules = association_rules(data, metric='lift', min_threshold=MIN_THRESH)
    
    # Sort and export
    rules.sort_values(by='lift', ascending=False)\
         .to_csv(f'results\\{name}_association_rule_report.csv', index=False)
    
    print(f"Exported: {name}_association_rule_report.csv")

Exported: comp_purchase_zero_association_rule_report.csv
Exported: comp_purchase_one_association_rule_report.csv
Exported: add_cart_zero_association_rule_report.csv
Exported: add_cart_one_association_rule_report.csv


In [9]:
'''
There's a strong disassociation between the products
meaning placing or selling products in bundle at C2C marketplace, will unlikely to be bought together
Some product are unlikely to be bought together
'''

"\nThere's a strong disassociation between the products\nmeaning placing or selling products in bundle at C2C marketplace, will unlikely to be bought together\nSome product are unlikely to be bought together\n\nBased on the zero category, deeper analysis is still required\n"